# Causal interventions — patching, ablation, steering

**Runtime → GPU (L4)**, set `ROLE` and `LANGUAGE` in cell 2, then **Run all**.

Cell 3 is a hard gate. It captures a layer's residual stream, writes the
identical values back, and asserts the logits do not move. **If that fails the
notebook stops**, because a mis-wired forward hook does not raise — it quietly
patches the wrong site and returns plausible numbers, which is the worst
possible failure for a causal claim. Nothing after cell 3 means anything until
cell 3 passes.

**Read the controls before the effects.** `ctrl-pos` is the same edit at random
non-variable positions. If it moves the metric as much as `intervened` does,
the result is "editing anything matters", not "this role matters".

**Case density is low** — per 300 programs: Python 184, Java 69, C++ 67,
PHP 45, JavaScript 34. Use the full split, and treat small effects with the
same suspicion the probing work earned: check them against the spread of the
controls before reading anything into them.

`boolean` yields ~2 cases per 300 programs because `pipeline/roles.py`'s
boolean extractor is very strict (issue #16). The other four roles are fine.


In [ ]:
# 1 - setup: clone/pull main, deps, token, restore prior work from Drive
PIN_COMMIT = ""
BRANCH = "main"
import os, pathlib
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin
if PIN_COMMIT:
    !git checkout -q {PIN_COMMIT}
else:
    !git checkout -q -B {BRANCH} origin/{BRANCH}
!git log --oneline -1
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-cpp>=0.23.4" \
  "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/causal"
!mkdir -p outputs/causal outputs/role_occ data/xlcost {DEST}
!cp -n {DEST}/*.json outputs/causal/ 2>/dev/null || true
print("setup complete")


In [ ]:
# 2 - CONFIG
ROLE = "accumulator"   # index_key | accumulator | iterator | boolean | class_struct
LANGUAGE = "Python"    # Python | Java | C++ | C | Javascript | PHP
SPLIT = "train"
MODELS = [
    "Qwen/Qwen2.5-Coder-1.5B",
    "Qwen/Qwen2.5-1.5B",      # base-vs-Coder contrast at identical scale
    "bigcode/starcoder2-7b",  # different family and pretraining corpus
]
slug = LANGUAGE.lower().replace("++", "pp").replace("#", "sharp")
mslug = lambda m: m.split("/")[-1].lower().replace(".", "").replace("-", "")
print(f"{ROLE} | {LANGUAGE}/{SPLIT} | {len(MODELS)} model(s)")


In [ ]:
# 3 - THE GATE. Logic self-check, then prove the hooks write where they claim.
#     This cell RAISES if either fails. Do not skip it and do not "just try
#     cell 4" -- a mis-wired hook returns plausible numbers silently.
import subprocess

r = subprocess.run(["python", "scripts/causal.py", "verify"],
                   capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    raise SystemExit("case-construction self-check FAILED - stop here")

for m in MODELS:
    print(f"\n######## GPU sanity: {m} ########")
    r = subprocess.run(["python", "scripts/causal.py", "sanity", "--model-id", m],
                       capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-2000:])
    if r.returncode != 0:
        raise SystemExit(
            f"HOOK SANITY FAILED for {m}. The self-patch no-op is the check that "
            "matters: if writing a layer's own captured values back changes the "
            "logits, the hook targets the wrong tensor, positions, or layer. "
            "Every causal number from this build would be noise. Stop."
        )
print("\nGATE PASSED - hooks verified, safe to run cell 4")


In [ ]:
# 4 - the sweep: patch + ablate + steer per model, with controls. GPU.
#     Checkpoints to Drive after each model so a disconnect cannot lose one.
for m in MODELS:
    print(f"\n############ {ROLE} / {LANGUAGE} - {m} ############")
    !bash scripts/run_causal.sh {ROLE} {LANGUAGE} {m} {SPLIT}
    !cp outputs/causal/{ROLE}_{slug}_{SPLIT}_{mslug(m)}_*.json {DEST}/ 2>/dev/null || true


In [ ]:
# 5 - results. Controls are printed BESIDE the effect on purpose: an effect
#     that its own random-position control matches is not a finding.
import glob, json

for mode in ("patch", "ablate", "steer"):
    print(f"\n=== {ROLE} / {LANGUAGE}/{SPLIT} - {mode} ===")
    for m in MODELS:
        f = f"outputs/causal/{ROLE}_{slug}_{SPLIT}_{mslug(m)}_{mode}.json"
        try:
            d = json.load(open(f))
        except FileNotFoundError:
            print(f"  {mslug(m):<22} (not run)"); continue
        print(f"  {mslug(m)}  cases={d['n_cases_scored']}/{d['n_cases_available']}"
              f"  commit={d.get('git_commit','?')[:12]}")
        if d.get("skipped"):
            print(f"    skipped: {d['skipped']}")
        print(f"    {'layer':>6}{'n':>5}{'clean':>9}{'interv':>9}{'effect':>9}"
              f"{'ctrl-pos':>10}{'ctrl-dir':>10}{'no-ctrl':>9}")
        for s in d["summary_by_layer"]:
            print(f"    {s['layer']:>6}{s['n']:>5}{s['clean_mean']:>9.3f}"
                  f"{s['intervened_mean']:>9.3f}{s['effect_mean']:>9.3f}"
                  f"{s.get('control_random_position_mean', float('nan')):>10.3f}"
                  f"{s.get('control_random_direction_mean', float('nan')):>10.3f}"
                  f"{s.get('n_without_positional_control', 0):>9}")
print("\nAn effect is only interpretable if it exceeds its own controls.")


In [ ]:
# 6 - save everything for this (role, language) to Drive
!mkdir -p {DEST}
!cp outputs/causal/{ROLE}_{slug}_{SPLIT}_*.json {DEST}/ 2>/dev/null || true
!cp outputs/role_occ/all_{slug}_{SPLIT}.jsonl* {DEST}/ 2>/dev/null || true
print(f"saved to {DEST}")
